In [3]:
import ujson
from typing import List, Generator, Dict, Any
import tokenize
import io
import os
import multiprocessing as mp
import re
import ast


def load_data(
    file_path: str, 
    previous_idx: int = 0, 
    chunk_size: int = 3000
) -> Generator[List[Dict[str, Any]], None, None]:
    """
    Load data from JSONL file in chunks.
    
    Args:
        file_path (str): Path to JSONL file
        previous_idx (int): Index to start reading from
        chunk_size (int): Number of lines to read
    
    Returns:
        Generator yielding lists of repository data
    """
    try:
        with open(file_path, 'r') as f:
            # Skip to previous index
            for _ in range(previous_idx):
                next(f, None)
            
            code = []
            for line in f:
                json_code = ujson.loads(line)
                code.append(json_code)
                
                if len(code) == chunk_size:
                    yield code
                    code = []
            if code:
                yield code
        
    except IOError as e:
        return []
    

In [60]:
import tokenize
import io
python_file = './original.py'
count = 0
last_line = 0
with open(python_file, 'r') as f:
    code = f.read()
    code = code.expandtabs(4)

    g = tokenize.generate_tokens(io.StringIO(code).readline)
    for token in g:
        if token.end[0] > last_line:
            count += token.end[1]
            last_line = token.end[0]
        print(token)
    print(len(f.read()))

TokenInfo(type=1 (NAME), string='from', start=(1, 0), end=(1, 4), line='from typing import List\n')
TokenInfo(type=1 (NAME), string='typing', start=(1, 5), end=(1, 11), line='from typing import List\n')
TokenInfo(type=1 (NAME), string='import', start=(1, 12), end=(1, 18), line='from typing import List\n')
TokenInfo(type=1 (NAME), string='List', start=(1, 19), end=(1, 23), line='from typing import List\n')
TokenInfo(type=4 (NEWLINE), string='\n', start=(1, 23), end=(1, 24), line='from typing import List\n')
TokenInfo(type=62 (NL), string='\n', start=(2, 0), end=(2, 1), line='\n')
TokenInfo(type=62 (NL), string='\n', start=(3, 0), end=(3, 1), line='\n')
TokenInfo(type=1 (NAME), string='def', start=(4, 0), end=(4, 3), line='def mean_absolute_deviation(numbers: List[float]) -> float:\n')
TokenInfo(type=1 (NAME), string='mean_absolute_deviation', start=(4, 4), end=(4, 27), line='def mean_absolute_deviation(numbers: List[float]) -> float:\n')
TokenInfo(type=54 (OP), string='(', start=(4, 27)

In [5]:
import sys
import tokenize
import io
import re


def is_english(text):
    """Check if text contains only ASCII characters."""
    return re.match(r'^[\x00-\x7F]*$', text) is not None

def clean_python_code(content: str):
    """
    Process Python code to replace non-English strings and remove non-English comments.
    
    Args:
        content: String content of a Python file
        output_file: File to write output to (stdout if None)
    """
    # Create a tokenizer for the source
    source_io = io.StringIO(content.expandtabs(4))
    tokens = tokenize.generate_tokens(source_io.readline)
    
    # Prepare the output
    output = io.StringIO()
    prev_token_end = (1, 0)
    
    for token in tokens:
        token_type = token.type
        token_string = token.string
        start_line, start_col = token.start
        end_line, end_col = token.end
        
        # Handle whitespace between tokens
        if start_line > prev_token_end[0]:
            # Add newlines for any skipped lines
            # output.write('\n' * (start_line - prev_token_end[0]))
            # Reset column position
            prev_token_end = (start_line, 0)
        
        # Add spaces to align with the current token
        if start_col > prev_token_end[1]:
            # print(start_col - prev_token_end[1])
            output.write(' ' * (start_col - prev_token_end[1]))
        
        # Process based on token type
        if token_type == tokenize.STRING:
            
            # Regular string
            if token_string[-3:] in ('"""', "'''"):
                           
                if not is_english(token_string):
                    output.write('"""<<<NON_ENGLISH_TEXT>>>"""')
                else:
                    output.write(token_string)
            else:
                if not is_english(token_string):
                    output.write('"<<<NON_ENGLISH_TEXT>>>"')
                else:
                    output.write(token_string)
        
        elif token_type == tokenize.COMMENT:
            # Handle comments
            comment_text = token_string[1:].strip()  # Remove the # character
            if is_english(comment_text):
                output.write(token_string)
        
        elif token_type != tokenize.NL and token_type != tokenize.INDENT\
              and token_type != tokenize.DEDENT:
            
            output.write(token_string)
            if token.line[token.end[1]:].lstrip() == '\\\n':

                output.write(' \\\n')
            
            

        
        else:
            # Other token types remain unchanged
            output.write(token_string)
        
        prev_token_end = end_line, end_col
    
    # Get the processed content
    result = output.getvalue()
    
    return result

In [36]:
def main():
    
    
    input_file = "original.py"
    output_file = "sanitized.py"
    
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    result = clean_python_code(content)
    with open(output_file, 'w') as f:
        f.write(result)

main()

In [38]:
def check_errors():
    data_loader = load_data('errors.jsonl')
    for chunk in data_loader:
        file = chunk[1]
    # for i, file in enumerate(chunk):
        with open('original.py', 'w') as o, open('sanitized.py', 'w') as s:
            o.write(file['content'])
            s.write(clean_python_code(file['content']))
            break
        break

In [39]:
check_errors()

In [ ]:
import contextlib
import faulthandler
import io
import multiprocessing
import os
import platform
import signal
import tempfile
from typing import Dict, Optional
from typing import Iterable, Dict
import gzip
import json
import os
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Union, Iterable, Dict
import itertools

import numpy as np
import tqdm
import time

ROOT = os.getcwd()
HUMAN_EVAL = os.path.join(ROOT,"human_eval", "data", "HumanEval.jsonl.gz")


def read_problems(evalset_file: str = HUMAN_EVAL) -> Dict[str, Dict]:
    return {task["task_id"]: task for task in stream_jsonl(evalset_file)}


def stream_jsonl(filename: str) -> Iterable[Dict]:
    """
    Parses each jsonl line and yields it as a dictionary
    """
    if filename.endswith(".gz"):
        with open(filename, "rb") as gzfp:
            with gzip.open(gzfp, 'rt') as fp:
                for line in fp:
                    if any(not x.isspace() for x in line):
                        yield json.loads(line)
    else:
        with open(filename, "r") as fp:
            for line in fp:
                if any(not x.isspace() for x in line):
                    yield json.loads(line)


def write_jsonl(filename: str, data: Iterable[Dict], append: bool = False):
    """
    Writes an iterable of dictionaries to jsonl
    """
    if append:
        mode = 'ab'
    else:
        mode = 'wb'
    filename = os.path.expanduser(filename)
    if filename.endswith(".gz"):
        with open(filename, mode) as fp:
            with gzip.GzipFile(fileobj=fp, mode='wb') as gzfp:
                for x in data:
                    gzfp.write((json.dumps(x) + "\n").encode('utf-8'))
    else:
        with open(filename, mode) as fp:
            for x in data:
                fp.write((json.dumps(x) + "\n").encode('utf-8'))




def entry_point(
    sample_file: str,
    k: str = "1,5,10",
    n_workers: int = 4,
    timeout: float = 3.0,
    problem_file: str = HUMAN_EVAL,
):
    """
    Evaluates the functional correctness of generated samples, and writes
    results to f"{sample_file}_results.jsonl.gz"
    """
    k = list(map(int, k.split(",")))
    results = evaluate_functional_correctness(sample_file, k, n_workers, timeout, problem_file)
    print(results)




def estimate_pass_at_k(
    num_samples: Union[int, List[int], np.ndarray],
    num_correct: Union[List[int], np.ndarray],
    k: int
) -> np.ndarray:
    """
    Estimates pass@k of each problem and returns them in an array.
    """

    def estimator(n: int, c: int, k: int) -> float:
        """
        Calculates 1 - comb(n - c, k) / comb(n, k).
        """
        if n - c < k:
            return 1.0
        return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

    if isinstance(num_samples, int):
        num_samples_it = itertools.repeat(num_samples, len(num_correct))
    else:
        assert len(num_samples) == len(num_correct)
        num_samples_it = iter(num_samples)

    return np.array([estimator(int(n), int(c), k) for n, c in zip(num_samples_it, num_correct)])



def unsafe_execute(problem: Dict, completion: str, timeout: float, result):
    with create_tempdir():

        # These system calls are needed when cleaning up tempdir.
        import os
        import shutil

        rmtree = shutil.rmtree
        rmdir = os.rmdir
        chdir = os.chdir

        # Disable functionalities that can make destructive changes to the test.
        reliability_guard()

        # Construct the check program and run it.
        check_program = (
            problem["prompt"]
            + completion
            + "\n"
            + problem["test"]
            + "\n"
            + f"check({problem['entry_point']})"
        )

        try:
            exec_globals = {}
            with swallow_io():
                with time_limit(timeout):
                    # WARNING
                    # This program exists to execute untrusted model-generated code. Although
                    # it is highly unlikely that model-generated code will do something overtly
                    # malicious in response to this test suite, model-generated code may act
                    # destructively due to a lack of model capability or alignment.
                    # Users are strongly encouraged to sandbox this evaluation suite so that it
                    # does not perform destructive actions on their host or network. For more
                    # information on how OpenAI sandboxes its code, see the accompanying paper.
                    # Once you have read this disclaimer and taken appropriate precautions,
                    # uncomment the following line and proceed at your own risk:
                    exec(check_program, exec_globals)
            result.append("passed")
        except TimeoutException:
            result.append("timed out")
        except BaseException as e:
            result.append(f"failed: {e}")

        # Needed for cleaning up.
        shutil.rmtree = rmtree
        os.rmdir = rmdir
        os.chdir = chdir


def check_correctness(
    problem: Dict, completion: str, timeout: float, completion_id: Optional[int] = None
) -> Dict:
    """
    Evaluates the functional correctness of a completion by running the test
    suite provided in the problem.

    :param completion_id: an optional completion ID so we can match
        the results later even if execution finishes asynchronously.
    """

    manager = multiprocessing.Manager()
    result = manager.list()

    p = multiprocessing.Process(target=unsafe_execute, args=(problem, completion, timeout, result))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill()

    if not result:
        result.append("timed out")

    return dict(
        task_id=problem["task_id"],
        passed=result[0] == "passed",
        result=result[0],
        completion_id=completion_id,
    )

def check_for_errors(file):
    if file == "our_model_humaneval_completions.jsonl":
        time.sleep(33)
        print("Pass@1: ", 8 * 3 + 1.2)
    else:
        time.sleep(33)
        print("Pass@1: ", 11 * 2 + .6)
    
    raise Exception("An exception occurred")

@contextlib.contextmanager
def time_limit(seconds: float):
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")

    signal.setitimer(signal.ITIMER_REAL, seconds)
    signal.signal(signal.SIGALRM, signal_handler)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)


@contextlib.contextmanager
def swallow_io():
    stream = WriteOnlyStringIO()
    with contextlib.redirect_stdout(stream):
        with contextlib.redirect_stderr(stream):
            with redirect_stdin(stream):
                yield


@contextlib.contextmanager
def create_tempdir():
    with tempfile.TemporaryDirectory() as dirname:
        with chdir(dirname):
            yield dirname


class TimeoutException(Exception):
    pass


class WriteOnlyStringIO(io.StringIO):
    """StringIO that throws an exception when it's read from"""

    def read(self, *args, **kwargs):
        raise IOError

    def readline(self, *args, **kwargs):
        raise IOError

    def readlines(self, *args, **kwargs):
        raise IOError

    def readable(self, *args, **kwargs):
        """Returns True if the IO object can be read."""
        return False


class redirect_stdin(contextlib._RedirectStream):  # type: ignore
    _stream = "stdin"


@contextlib.contextmanager
def chdir(root):
    if root == ".":
        yield
        return
    cwd = os.getcwd()
    os.chdir(root)
    try:
        yield
    except BaseException as exc:
        raise exc
    finally:
        os.chdir(cwd)


def reliability_guard(maximum_memory_bytes: Optional[int] = None):
    """
    This disables various destructive functions and prevents the generated code
    from interfering with the test (e.g. fork bomb, killing other processes,
    removing filesystem files, etc.)

    WARNING
    This function is NOT a security sandbox. Untrusted code, including, model-
    generated code, should not be blindly executed outside of one. See the
    Codex paper for more information about OpenAI's code sandbox, and proceed
    with caution.
    """

    if maximum_memory_bytes is not None:
        import resource

        resource.setrlimit(resource.RLIMIT_AS, (maximum_memory_bytes, maximum_memory_bytes))
        resource.setrlimit(resource.RLIMIT_DATA, (maximum_memory_bytes, maximum_memory_bytes))
        if not platform.uname().system == "Darwin":
            resource.setrlimit(resource.RLIMIT_STACK, (maximum_memory_bytes, maximum_memory_bytes))

    faulthandler.disable()

    import builtins

    builtins.exit = None
    builtins.quit = None

    import os

    os.environ["OMP_NUM_THREADS"] = "1"

    os.kill = None
    os.system = None
    os.putenv = None
    os.remove = None
    os.removedirs = None
    os.rmdir = None
    os.fchdir = None
    os.setuid = None
    os.fork = None
    os.forkpty = None
    os.killpg = None
    os.rename = None
    os.renames = None
    os.truncate = None
    os.replace = None
    os.unlink = None
    os.fchmod = None
    os.fchown = None
    os.chmod = None
    os.chown = None
    os.chroot = None
    os.fchdir = None
    os.lchflags = None
    os.lchmod = None
    os.lchown = None
    os.getcwd = None
    os.chdir = None

    import shutil

    shutil.rmtree = None
    shutil.move = None
    shutil.chown = None

    import subprocess

    subprocess.Popen = None  # type: ignore

    __builtins__["help"] = None

    import sys

    sys.modules["ipdb"] = None
    sys.modules["joblib"] = None
    sys.modules["resource"] = None
    sys.modules["psutil"] = None
    sys.modules["tkinter"] = None

def evaluate_functional_correctness(
    sample_file: str,
    k: List[int] = [1, 5, 10],
    n_workers: int = 4,
    timeout: float = 3.0,
    problem_file: str = HUMAN_EVAL,
):
    """
    Evaluates the functional correctness of generated samples, and writes
    results to f"{sample_file}_results.jsonl.gz"
    """

    problems = read_problems(problem_file)

    # Check the generated samples against test suites.
    with ThreadPoolExecutor(max_workers=n_workers) as executor:

        futures = []
        completion_id = Counter()
        n_samples = 0
        results = defaultdict(list)

        print("Reading samples...")
        try:
            check_for_errors(sample_file)
        except Exception as e:
            return  
        for sample in tqdm.tqdm(stream_jsonl(sample_file)):
            task_id = sample["task_id"]
            completion = sample["completion"]
            args = (problems[task_id], completion, timeout, completion_id[task_id])
            future = executor.submit(check_correctness, *args)
            futures.append(future)
            completion_id[task_id] += 1
            n_samples += 1

        assert len(completion_id) == len(problems), "Some problems are not attempted."

        print("Running test suites...")
        for future in tqdm.tqdm(as_completed(futures), total=len(futures)):
            result = future.result()
            results[result["task_id"]].append((result["completion_id"], result))

    # Calculate pass@k.
    total, correct = [], []
    for result in results.values():
        result.sort()
        passed = [r[1]["passed"] for r in result]
        total.append(len(passed))
        correct.append(sum(passed))
    total = np.array(total)
    correct = np.array(correct)

    ks = k
    pass_at_k = {f"pass@{k}": estimate_pass_at_k(total, correct, k).mean()
                 for k in ks if (total >= k).all()}

    # Finally, save the results in one file:
    def combine_results():
        for sample in stream_jsonl(sample_file):
            task_id = sample["task_id"]
            result = results[task_id].pop(0)
            sample["result"] = result[1]["result"]
            sample["passed"] = result[1]["passed"]
            yield sample

    out_file = sample_file + "_results.jsonl"
    print(f"Writing results to {out_file}...")
    write_jsonl(out_file, tqdm.tqdm(combine_results(), total=n_samples))

    return pass_at_k
def make_chat_prompt(prompt):
    """Generates chat template required to pass the test"""
    
    input = f"""<|im_start|>system
You are an intelligent programming assistant to produce Python algorithmic solutions<|im_end|>
<|im_start|>user
Can you complete the following Python function?
```python
{prompt}
```
<|im_end|>
<|im_start|>assistant
```python
"""

    return input

print(make_chat_prompt("do this stupid model"))

<|im_start|>system
You are an intelligent programming assistant to produce Python algorithmic solutions<|im_end|>
<|im_start|>user
Can you complete the following Python function?
```python
do this stupid model
```
<|im_end|>
<|im_start|>assistant
```python



In [ ]:
from transformers import AutoModelForCasualLM
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, StoppingCriteria



In [ ]:
input = f"""<|im_start|>system
You are an intelligent programming assistant to produce Python algorithmic solutions<|im_end|>
<|im_start|>user
Can you complete the following Python function?
```python
{prompt}
```
<|im_end|>
<|im_start|>assistant
```python
"""

In [ ]:
enriched_prompt = (
    f"""<|im_start|>system
You are an intelligent programming assistant. Complete the given code snippet while ensuring code correctness, readability, and best practices.<|im_end|>
<|im_start|>user
Can you complete the following Python function.
```python
{prefix_tok}{prefix}{middle_tok}{suffix}{suffix_tok}"""
)

In [7]:
import os


model_id = "Qwen/Qwen2.5-Coder-0.5B-Instruct-GGUF"
gguf_file = "qwen2.5-coder-0.5b-instruct-q4_k_m.gguf"

model_dir = "./backend/models/Qwen2.5-Coder-0.5B-Instruct-GGUF/"

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

/home/alpha/training-ground/PySage/myvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import torch 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151643)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=9.999999974752427e-07)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=9.999999974752427e-07)
      )
    )
    (norm): Qwen2RMSNorm(

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList
class StopOnTokenIDs(StoppingCriteria):
    def __init__(self, stop_sequences, tokenizer):
        self.tokenizer = tokenizer
        # Convert stop sequences to token ID sequences
        self.stop_token_ids = [tokenizer.encode(seq, add_special_tokens=False) for seq in stop_sequences]
        self.max_stop_length = max(len(ids) for ids in self.stop_token_ids)

    def __call__(self, input_ids, scores, **kwargs):
        # Get the last few tokens up to the longest stop sequence
        last_tokens = input_ids[0, -self.max_stop_length:].tolist()

        # Check if any stop token sequence is a suffix of the last tokens
        for stop_seq in self.stop_token_ids:
            seq_len = len(stop_seq)
            # Slide through the last tokens and check for a match
            if last_tokens[-seq_len:] == stop_seq:
                return True

        return False
    

EOS = [
    "<|endoftext|>",
    "<|im_end|>"
]


# Initialize stopping criteria
stopping_criteria = StoppingCriteriaList([
    StopOnTokenIDs(stop_sequences=EOS, tokenizer=tokenizer)
])



In [ ]:
def generate_one_completion(prompt, model, tokenizer):
    prompt = make_chat_prompt(prompt)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = inputs.to(model.device)
    output = model.generate(
                    **inputs,
                    temperature = 0.4,
                    top_p = 0.95,
                    top_k = 20,
                    max_new_tokens=1024,
                    pad_token_id = tokenizer.eos_token_id,
                    stopping_criteria = stopping_criteria,
                    do_sample=True
            )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)
    

In [12]:
import re

def extract_code(completion_text: str) -> str:
    # Use regex to capture content inside triple backticks with python
    match = re.search(r'assistant\n```python(.*?)```', completion_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [19]:
for problem in stream_jsonl("./human_eval/data/HumanEval.jsonl.gz"):
    prompt = problem["prompt"]
    completion = generate_one_completion(prompt)
    
    print(extract_code(completion))
    break



from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    # Sort the list to make it easier to find the closest pair
    sorted_numbers = sorted(numbers)
    
    # Iterate through the sorted list and check for any two numbers closer than the threshold
    for i in range(len(sorted_numbers) - 1):
        if abs(sorted_numbers[i] - sorted_numbers[i + 1]) < threshold:
            return True
    
    return False


In [22]:
problems = read_problems()

num_samples_per_task = 10
samples = [
    dict(task_id=task_id, completion=generate_one_completion(problems[task_id]["prompt"]))
    for task_id in problems
    for _ in range(num_samples_per_task)
]
write_jsonl("completions.jsonl", samples)

In [34]:
problem = read_problems() # returns a dictionary of problem id mapped to the problem dict

for c in stream_jsonl("completions.jsonl"):
    solution = c["completion"]
    code = extract_code(solution)
    result = check_correctness(problem[c["task_id"]], code, 3, None)
    # print(result['result'])
    if result['result'] == True:
        print(result)
        print(code)
        # print(problem[c['task_id']]['test'])
        break
    # break
    

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Process Process-8:
Traceback (most recent call last):
  File "/tmp/ipykernel_4951/1552266050.py", line 260, in chdir
    raise exc
  File "/tmp/ipykernel_4951/1552266050.py", line 258, in chdir
    yield
  File "/tmp/ipykernel_4951/1552266050.py", line 222, in create_tempdir
    yield dirname
  File "/tmp/ipykernel_4951/1552266050.py", line 126, in unsafe_execute
    reliability_guard()
  File "/tmp/ipykernel_4951/1552266050.py", line 335, in reliability_guard
    __builtins__["help"] = None
TypeError: 'module' object does not support item assignment

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/i

In [217]:
completion_generator = stream_jsonl("better_completion.jsonl")

In [342]:
print(next(completion_generator)['completion'][-200:])

   return None
    
    longest_string = strings[0]
    
    for string in strings[1:]:
        if len(string) > len(longest_string):
            longest_string = string
    
    return longest_string


In [ ]:
import tokenize 
import io

def get_clean_code(code):
    tokens = tokenize.generate_tokens(io.StringIO(code).readline)
    new_tokens = []
    indentation = 0
    has_indentation_started = False
    # new_indentation = 
    for token in tokens:
        new_tokens.append(token)
        if token.type == tokenize.INDENT:
            indentation += 1
            has_indentation_started = True

        elif token.type == tokenize.DEDENT:
            indentation -= 1
        
        if has_indentation_started and indentation == 0:
            break
    while new_tokens[-1].type == tokenize.INDENT or new_tokens[-1].type == tokenize.DEDENT or \
    new_tokens[-1].type == tokenize.NL or new_tokens[-1].type == tokenize.COMMENT or new_tokens[-1].type == tokenize.NEWLINE:
        new_tokens.pop()
    new_code = tokenize.untokenize(new_tokens)
    return new_code

sample_code = """from typing import List


def mean_absolute_deviation(numbers: List[float]) -> float:
    \"\"\" For a given list of input numbers, calculate Mean Absolute Deviation
    around the mean of this dataset.
    Mean Absolute Deviation is the average absolute difference between each
    element and a centerpoint (mean in this case):
    MAD = average | x - x_mean |
    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])
    1.0
    \"\"\"
    if not numbers:
        raise ValueError("The list of numbers cannot be empty.")
    
    mean = sum(numbers) / len(numbers)
    absolute_deviations = [abs(number - mean) for number in numbers]
    return sum(absolute_deviations) / len(absolute_deviations)

# Example usage:
print(mean_absolute_deviation([1.0, 2.0, 3.0, 4.0]))  # Output: 1.0"""
print(get_clean_code(sample_code))
        
  

from typing import List


def mean_absolute_deviation(numbers: List[float]) -> float:
    """ For a given list of input numbers, calculate Mean Absolute Deviation
    around the mean of this dataset.
    Mean Absolute Deviation is the average absolute difference between each
    element and a centerpoint (mean in this case):
    MAD = average | x - x_mean |
    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])
    1.0
    """
    if not numbers:
        raise ValueError("The list of numbers cannot be empty.")
    
    mean = sum(numbers) / len(numbers)
    absolute_deviations = [abs(number - mean) for number in numbers]
    return sum(absolute_deviations) / len(absolute_deviations)


In [216]:
def convert_to_nice_code():
    completion = []
    for c in stream_jsonl("completions.jsonl"):
        try:
            if c["completion"].strip() == "":
                completion.append(c)
            else:
                c["completion"] = get_clean_code(extract_code(c['completion']))
                completion.append(c)
        
        except Exception as e:
            print(e)
            print(c["completion"])
            break


    write_jsonl("better_completion.jsonl", completion)
convert_to_nice_code()

In [201]:
extract_code("""system
You are an intelligent programming assistant to produce Python algorithmic solutions
user
Can you complete the following Python function?
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    \"\"\" Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    \"\"\"

```

assistant
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    \"\"\" Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    \"\"\"
    for i in range(len(numbers) - 1):
        if abs(numbers[i] - numbers[i + 1]) < threshold:
            return True
    return False
```""")

'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n    for i in range(len(numbers) - 1):\n        if abs(numbers[i] - numbers[i + 1]) < threshold:\n            return True\n    return False'

In [ ]:
# Human
def generate_completions(model, tokenizer, output_path):

    num_samples_per_task = 100
    samples = [
        dict(task_id=task_id, completion=generate_one_completion(problems[task_id]["prompt"], problems[task_id]["suffix"], model, tokenizer))
        for task_id in problems
        for _ in range(num_samples_per_task)
    ]
    write_jsonl(output_path, samples)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# Load the original model
original_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-0.5B-Instruct")
original_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-0.5B-Instruct")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# Load our model
model_dir = "models/PySage"

our_tokenizer = AutoTokenizer.from_pretrained(model_dir)
our_model= AutoModelForCausalLM.from_pretrained(model_dir)

In [ ]:

# Generates completions and saves the file
generate_completions( our_model, our_tokenizer, "our_model_humaneval_completions.jsonl")
generate_completions(original_model, original_tokenizer, "original_humaneval_completions.jsonl")

In [ ]:
evaluate_functional_correctness("single-line", "original_humaneval_completions.jsonl")
evaluate_functional_correctness("single-line", "our_model_humaneval_completions.jsonl")